In [0]:
CREATE WIDGET TEXT end_date DEFAULT '2025-11-30';

# 5yr Dx + 2yr Tx

In [0]:
CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

-- Medical events Dx (NPI = COALESCE(rendering, referring))
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'

UNION

-- Pharmacy events Dx (NPI = prescriber_npi; paid only)
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_STATUS = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}';


-- =============================================================================
-- STEP 2: TREATMENT CLAIMS (Tx universe for visit counting)
--   • Captures Elaprase-coded treatment from medical NDC, medical procedures, and paid pharmacy NDC
--   • Window: 2020-08-01 → ${end_date}
--   • Includes CODE field to retain NDC/procedure provenance
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW all_tx_claims_2yr AS

-- Medical events Tx via NDC (NPI = COALESCE(rendering, referring))
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'

UNION

-- Medical events Tx via procedures (NPI = rendering_npi only)
SELECT DISTINCT
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    PROCEDURE_CODE AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743',
                         'S9357', 'S9379', '38206', '38230', '38232',
                         '38240', '38241', '38242', '38243', '38250')
  AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'

UNION

-- Pharmacy events Tx via NDC (NPI = prescriber_npi; paid only)
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}';

-- 4A) Specified Dx requirement: ≥2 distinct E761 Dx dates in Dx window
CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

-- 4B) Specified cohort: specified Dx + any Tx in 2y/refresh window
CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN all_tx_claims_2yr t ON e.PATIENT_ID = t.PATIENT_ID;

-- 4C) Incremental Dx requirement: ≥2 distinct E763 Dx dates in Dx window
CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

-- 4D) Elaprase-coded Tx requirement for incremental eligibility (2y/refresh window)
CREATE OR REPLACE TEMPORARY VIEW elaprase_tx_2yr AS
SELECT DISTINCT PATIENT_ID
FROM all_tx_claims_2yr
WHERE CODE IN ('54092070001', '540920700', 'J1743');

-- 4E) Incremental cohort: incremental Dx + Elaprase-coded tx in 2y window + exclude specified
CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN elaprase_tx_2yr t ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);

-- 4F) Final eligible cohort
CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;


CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS
SELECT *
FROM (
  -- Dx claims
  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_dx_claims
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

  UNION

  -- Tx claims
  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_tx_claims_2yr
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
);

In [0]:
select count(distinct a.patient_id) as patient_counts, count(distinct a.npi) as hcp_counts, count(distinct b.hco_npi) as hco_counts
from all_patient_claims as a
left join com_edp_prd.cmpa_insights_internal_schema.reference_file_0109 as b on a.npi = b.hcp_npi

In [0]:
CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS
WITH hcp_metrics AS (
    SELECT
        a.PATIENT_ID,
        a.NPI,

        -- Specialty bucket (reporting label)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 'Geneticist'
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 'Psychiatry & Neurology'
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 'Pediatrician'
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 'PCP'
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 'NPPA'
            WHEN a.NPI IS NULL
                THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY,

        -- Tier 1: specialty priority (lower is better)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 5
            WHEN a.NPI IS NULL
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,

        -- Tier 2: visit counts (Dx + Tx combined)
        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

        -- Reference-only breakdowns (not used in ranking)
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,

        -- Tier 3: most recent visit date
        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p
        ON a.NPI = p.NPI
    GROUP BY
        a.PATIENT_ID,
        a.NPI,
        p.primary_specialty,
        p.secondary_specialty
),
ranked_hcps AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY
                SPECIALTY_PRIORITY ASC,
                NO_OF_VISITS DESC,
                MOST_RECENT_VISIT DESC,
                NPI ASC
        ) AS HCP_RANK
    FROM hcp_metrics
)
SELECT
    PATIENT_ID,
    NPI AS PRIMARY_HCP_NPI,
    SPECIALTY AS PRIMARY_HCP_SPECIALTY,
    SPECIALTY_PRIORITY,
    NO_OF_VISITS,
    DX_VISITS,
    TX_VISITS,
    MOST_RECENT_VISIT,
    HCP_RANK
FROM ranked_hcps
WHERE HCP_RANK = 1;

In [0]:
create or replace temp view primary_hcp_hco as (
  select 
a.patient_id,
a.primary_hcp_npi as hcp_npi,
b.hco_npi
from primary_hcp a
left join com_edp_prd.cmpa_insights_internal_schema.reference_file_0109 b on a.primary_hcp_npi = b.hcp_npi)

In [0]:
select count(distinct patient_id) as patient_counts, count(distinct hcp_npi) as hcp_counts, count(distinct hco_npi) as hco_counts
from primary_hcp_hco

# 3yr Dx + 1yr Tx

In [0]:
CREATE OR REPLACE TEMPORARY VIEW all_dx_claims_3yr AS

SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2022-08-01' AND '${end_date}'

UNION

SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_STATUS = 'PAID'
  AND FILL_DATE BETWEEN '2022-08-01' AND '${end_date}';


CREATE OR REPLACE TEMPORARY VIEW all_tx_claims_1yr AS

SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND SERVICE_DATE BETWEEN '2024-08-01' AND '${end_date}'

UNION

SELECT DISTINCT
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    PROCEDURE_CODE AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743',
                         'S9357', 'S9379', '38206', '38230', '38232',
                         '38240', '38241', '38242', '38243', '38250')
  AND SERVICE_DATE BETWEEN '2024-08-01' AND '${end_date}'

UNION

SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2024-08-01' AND '${end_date}';


CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2022-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2022-08-01' AND '${end_date}'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN all_tx_claims t ON e.PATIENT_ID = t.PATIENT_ID;

CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2022-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2022-08-01' AND '${end_date}'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

CREATE OR REPLACE TEMPORARY VIEW elaprase_tx AS
SELECT DISTINCT PATIENT_ID
FROM all_tx_claims_1yr
WHERE CODE IN ('54092070001', '540920700', 'J1743');

CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN elaprase_tx t ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);

CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;

CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS
  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_dx_claims_3yr
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

  UNION

  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_tx_claims_1yr
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients);

In [0]:
select count(distinct a.patient_id) as patient_counts, count(distinct a.npi) as hcp_counts, count(distinct b.hco_npi) as hco_counts
from all_patient_claims as a
left join com_edp_prd.cmpa_insights_internal_schema.reference_file_0109 as b on a.npi = b.hcp_npi

In [0]:
CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS
WITH hcp_metrics AS (
    SELECT
        a.PATIENT_ID,
        a.NPI,

        -- Specialty bucket (reporting label)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 'Geneticist'
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 'Psychiatry & Neurology'
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 'Pediatrician'
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 'PCP'
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 'NPPA'
            WHEN a.NPI IS NULL
                THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY,

        -- Tier 1: specialty priority (lower is better)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 5
            WHEN a.NPI IS NULL
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,

        -- Tier 2: visit counts (Dx + Tx combined)
        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

        -- Reference-only breakdowns (not used in ranking)
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,

        -- Tier 3: most recent visit date
        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p
        ON a.NPI = p.NPI
    GROUP BY
        a.PATIENT_ID,
        a.NPI,
        p.primary_specialty,
        p.secondary_specialty
),
ranked_hcps AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY
                SPECIALTY_PRIORITY ASC,
                NO_OF_VISITS DESC,
                MOST_RECENT_VISIT DESC,
                NPI ASC
        ) AS HCP_RANK
    FROM hcp_metrics
)
SELECT
    PATIENT_ID,
    NPI AS PRIMARY_HCP_NPI,
    SPECIALTY AS PRIMARY_HCP_SPECIALTY,
    SPECIALTY_PRIORITY,
    NO_OF_VISITS,
    DX_VISITS,
    TX_VISITS,
    MOST_RECENT_VISIT,
    HCP_RANK
FROM ranked_hcps
WHERE HCP_RANK = 1;

In [0]:
create or replace temp view primary_hcp_hco as (
  select 
a.patient_id,
a.primary_hcp_npi as hcp_npi,
b.hco_npi
from primary_hcp a
left join com_edp_prd.cmpa_insights_internal_schema.reference_file_0109 b on a.primary_hcp_npi = b.hcp_npi)

In [0]:
select count(distinct patient_id) as patient_counts, count(distinct hcp_npi) as hcp_counts, count(distinct hco_npi) as hco_counts
from primary_hcp_hco

# 2yr Dx + 1yr Tx

In [0]:
CREATE OR REPLACE TEMPORARY VIEW all_dx_claims_2yr AS

SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'

UNION

SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_STATUS = 'PAID'
  AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}';


CREATE OR REPLACE TEMPORARY VIEW all_tx_claims_1yr AS

SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND SERVICE_DATE BETWEEN '2024-08-01' AND '${end_date}'

UNION

SELECT DISTINCT
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    PROCEDURE_CODE AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743',
                         'S9357', 'S9379', '38206', '38230', '38232',
                         '38240', '38241', '38242', '38243', '38250')
  AND SERVICE_DATE BETWEEN '2024-08-01' AND '${end_date}'

UNION

SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2024-08-01' AND '${end_date}';


CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN all_tx_claims t ON e.PATIENT_ID = t.PATIENT_ID;

CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

CREATE OR REPLACE TEMPORARY VIEW elaprase_tx AS
SELECT DISTINCT PATIENT_ID
FROM all_tx_claims_1yr
WHERE CODE IN ('54092070001', '540920700', 'J1743');

CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN elaprase_tx t ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);

CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;

CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS
  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_dx_claims_2yr
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

  UNION

  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_tx_claims_1yr
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients);

In [0]:
select count(distinct a.patient_id) as patient_counts, count(distinct a.npi) as hcp_counts, count(distinct b.hco_npi) as hco_counts
from all_patient_claims as a
left join com_edp_prd.cmpa_insights_internal_schema.reference_file_0109 as b on a.npi = b.hcp_npi

In [0]:
CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS
WITH hcp_metrics AS (
    SELECT
        a.PATIENT_ID,
        a.NPI,

        -- Specialty bucket (reporting label)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 'Geneticist'
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 'Psychiatry & Neurology'
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 'Pediatrician'
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 'PCP'
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 'NPPA'
            WHEN a.NPI IS NULL
                THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY,

        -- Tier 1: specialty priority (lower is better)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 5
            WHEN a.NPI IS NULL
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,

        -- Tier 2: visit counts (Dx + Tx combined)
        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

        -- Reference-only breakdowns (not used in ranking)
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,

        -- Tier 3: most recent visit date
        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p
        ON a.NPI = p.NPI
    GROUP BY
        a.PATIENT_ID,
        a.NPI,
        p.primary_specialty,
        p.secondary_specialty
),
ranked_hcps AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY
                SPECIALTY_PRIORITY ASC,
                NO_OF_VISITS DESC,
                MOST_RECENT_VISIT DESC,
                NPI ASC
        ) AS HCP_RANK
    FROM hcp_metrics
)
SELECT
    PATIENT_ID,
    NPI AS PRIMARY_HCP_NPI,
    SPECIALTY AS PRIMARY_HCP_SPECIALTY,
    SPECIALTY_PRIORITY,
    NO_OF_VISITS,
    DX_VISITS,
    TX_VISITS,
    MOST_RECENT_VISIT,
    HCP_RANK
FROM ranked_hcps
WHERE HCP_RANK = 1;

In [0]:
create or replace temp view primary_hcp_hco as (
  select 
a.patient_id,
a.primary_hcp_npi as hcp_npi,
b.hco_npi
from primary_hcp a
left join com_edp_prd.cmpa_insights_internal_schema.reference_file_0109 b on a.primary_hcp_npi = b.hcp_npi)

In [0]:
select count(distinct patient_id) as patient_counts, count(distinct hcp_npi) as hcp_counts, count(distinct hco_npi) as hco_counts
from primary_hcp_hco